In [13]:
# @title Instalar dependencias

# Instalar PyTorch y bibliotecas compatibles (CUDA 12.6)
!pip install --quiet torch==2.9.0 torchvision==0.24.0+cu126 torchaudio==2.9.0+cu126 --index-url https://download.pytorch.org/whl/cu126

# Transformers y utilidades
!pip install --quiet transformers rapidfuzz unidecode tqdm

# POS/Lemma (opcional)
!pip install --quiet spacy
!python -m spacy download es_core_news_sm --quiet

# @title Instalar dependencias
#!pip -q install transformers torch --upgrade
#!pip -q install rapidfuzz unidecode tqdm

# POS/Lemma (opcional).
#!pip -q install spacy
#!python -m spacy download es_core_news_sm -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 33.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [14]:
# @title Configuración de rutas y parámetros del sistema
from pathlib import Path

# Rutas
RUTA_XML_SALUD      = "/content/oraciones_salud_final.xml/".rstrip("/")
RUTA_XML_EDUCACION  = "/content/oraciones_educacion_final.xml/".rstrip("/")
RUTA_XML_CULTURA    = "/content/oraciones_cultura_final.xml/".rstrip("/")
RUTA_PICTOS         = "/content/todos_pictogramas/".rstrip("/")

# Salidas
OUT_SALUD      = "/content/candidatos_salud_final.json"
OUT_EDUCACION  = "/content/candidatos_educacion_final.json"
OUT_CULTURA    = "/content/candidatos_cultura_final.json"

# Cache de embeddings de etiquetas
CACHE_EMB_PICTOS = "/content/emb_pictos_v3.npy"

# Modelo
MODEL_NAME = "dccuchile/bert-base-spanish-wwm-cased"

# Parámetros de ranking (estrictos)
K_POR_TOKEN   = 6          # Top-K finales por token
M_POOL        = 60         # Pool léxico inicial
UMBRAL_SCORE  = 0.52       # Si top-1 < UMBRAL_SCORE => se omite/

# Reglas para tokens cortos / fuzzy
LONG_MIN_FUZZY      = 6    # No usar fuzzy si len(token) < 6 (evita piano->pino, etc.)
TOKEN_CORTO_EXACTO  = 3    # Si len(token) <= 3, SOLO exacto/sinónimo

# Stopwords y puntuación
OMITIR_STOPWORDS_EN_SALIDA = False   # True = NO incluirlas en el JSON (salvo allowlist forzado)
STOP_TOKENS = set("""
 el las un para
, . ; : ! ? ¿ ¡ ( ) - ' " _
""".split())

# --- Allowlist de stopwords que SÍ son importantes de exportar (con o sin candidatos) ---
ALLOWLIST_STOPWORDS_RAW = {
    'a','ahí','al','algunos','ante','antes','aquel','aquellos','aquí','aunque',
    'ayer','bajo','como','con','contra','cuando','cuándo','cómo','de','del',
    'demasiado','desde','después','durante','dónde','e','ella','ellas','ellos',
    'en','entre','esos','estar','este','estos','haber','hacia','hasta','hoy',
    'la','los','mañana','más','menos','mis','mucho','muchos','muy','ni','ninguno',
    'ningún','no','nosotras','nosotros','nuestro','nuestros','nunca','o','otro',
    'otros','para','pero','poco','pocos','por','porque','que','qué','quién',
    'quiénes','según','ser','seria','si','sí','siempre','sin','sobre','su','sus',
    'tantos','todo','todos','tras','tu','tus','tú','una','unas','uno','unos',
    'vía','vosotras','vosotros','vuestro','vuestros','y','ya','yo'
}
# La versión normalizada se crea en la Celda 3 (tras definir norm_text)
ALLOWLIST_STOPWORDS = None

# Si una palabra está en la allowlist y NO tiene pictogramas, la incluimos igual con "" TRUE para incluir
FORZAR_INCLUSION_ALLOWLIST_VACIOS = True

# Sinónimos base
SINONIMOS_BASE = {
    "baño": {"banio","bano","wc","aseo","lavabo","servicio","sanitario","vater"},
    "dolor": {"doler","molestia"},
    "maestro": {"profesor","profe","docente"},
    "tarea": {"deberes","trabajo_escolar"},
    "comida": {"alimentacion","alimento"},
    "escuela": {"colegio"},
    "hablar": {"platicar"},
    "enfermera": {"enfermero"},
    "doctor": {"medico"},
    # (Opcional) empuje para permitidas cortas
    "si": {"sí"},
    "no": {"prohibido"},
    "hoy": {"dia_de_hoy"},
    "manana": {"mañana"},   # la normalización ya equipara
    "ayer": {"dia_de_ayer"},
    "aqui": {"aqui","aquí"},
    "ahi": {"ahi","ahí"},
    "donde": {"donde","dónde"},
    "cuando": {"cuando","cuándo"},
    "como": {"como","cómo"},
    "que": {"que","qué"},
    "quien": {"quien","quién"},
}

# Semillas de n-gramas (se priorizan si existen en BD)
SEMILLAS_NGRAM = {
    "por favor","buenos dias","buenas tardes","buenas noches",
    "cuarto de baño","lavar las manos","tomar medicina","me duele",
    "no entiendo","necesito ayuda","quiero ir","quiero agua",
}

# Filtro de contenidos sensibles (por dominio y global)
BLOCKLIST_GLOBAL = {"genital","porn","violac","violación","estupro","mastur","erec","orgas","coito","sexo"}
BLOCKLIST_POR_DOMINIO = {
    "educacion": BLOCKLIST_GLOBAL | {"alcohol","droga"},
    "cultura":   BLOCKLIST_GLOBAL,
    "salud":     BLOCKLIST_GLOBAL,
}

# Boost por dominio (palabras que suben score si aparecen en la etiqueta)
BOOST_DOMINIO = {
    "educacion": {"escuela","clase","colegio","maestro","profesor","tarea","recreo","examen","libro","cuaderno"},
    "cultura":   {"baile","mariachi","comida","tortilla","fiesta","tradicion","museo","danza","musica","artesania"},
    "salud":     {"dolor","medico","enfermera","hospital","curita","termometro","inyeccion","medicina","sangre","vendaje"},
}

# Coherencia contextual (bonos)
W_COOCC = 0.06      # bono por co-ocurrencia con otras palabras de la oración
W_BIGRAM = 0.08     # bono si la etiqueta contiene un bigrama relevante de la oración
W_BOOST_DOM = 0.03  # bono por palabras de dominio

# --- Bonos/penalizaciones para asegurar unigrama exacto ---
EXACT_UNIGRAM_BONUS      = 3.5   # empuja exactos (e.g., "agua" == "agua")
UNIGRAM_PREFER_BONUS     = 0.8   # preferir nombres de 1 palabra
HEAD_POS_BONUS           = 0.25  # si el token está al inicio/fin del compuesto
LONG_COMPOUND_PENALTY    = 0.40  # penaliza >=3 tokens
ADJ_VERB_NOUN_PAIR_BONUS = 0.15  # bonus para pares verbo+sustantivo frecuentes


In [15]:
# @title Utilidades de normalización, lectura, lemas y POS
import os, re, json, numpy as np
import xml.etree.ElementTree as ET
from collections import defaultdict
from typing import List, Dict, Tuple, Set
from unidecode import unidecode
from rapidfuzz import fuzz

# spaCy
USE_SPACY = True
nlp = None
if USE_SPACY:
    try:
        import spacy
        nlp = spacy.load("es_core_news_sm")
    except Exception as e:
        print("spaCy no disponible. Continuando sin lemas/POS. Detalle:", e)
        nlp = None

def norm_text(s: str) -> str:
    """ Normaliza texto: minúsculas, sin tildes, espacios simples. """
    s = s.strip().lower().replace("_", " ")
    s = re.sub(r"\s+", " ", s)
    s = unidecode(s)
    return s

# --- Construimos la allowlist normalizada ahora que norm_text existe ---
global ALLOWLIST_STOPWORDS
ALLOWLIST_STOPWORDS = { norm_text(w) for w in ALLOWLIST_STOPWORDS_RAW }

def es_puntuacion(token: str) -> bool:
    # True si es solo signos/puntuación (para ignorar)
    return bool(re.fullmatch(r"[\W_]+", token))

def es_stop(token: str) -> bool:
    t = norm_text(token)
    # Es stop sólo si está en STOP_TOKENS y NO está en la allowlist
    return (t in STOP_TOKENS) and (t not in ALLOWLIST_STOPWORDS)

def es_stop_or_punt(token: str) -> bool:
    return es_puntuacion(token) or es_stop(token)

def tokens_de_etiqueta(et_norm: str) -> List[str]:
    """ Tokeniza etiquetas normalizadas en [a-z0-9]+ (palabras). """
    return re.findall(r"[a-z0-9]+", et_norm)

def leer_oraciones_xml(path_xml: str) -> List[str]:
    """ Lee <frase> del XML y devuelve lista de oraciones. """
    tree = ET.parse(path_xml)
    root = tree.getroot()
    frases = []
    for fr in root.iter("frase"):
        texto = (fr.text or "").strip()
        texto = re.sub(r"\s+", " ", texto)
        if texto:
            frases.append(texto)
    return frases

def lematiza_tokens(tokens: List[str]) -> List[str]:
    """ Devuelve lemmas si spaCy está disponible; si no, devuelve los mismos tokens. """
    if nlp is None:
        return tokens
    doc = nlp(" ".join(tokens))
    lem = []
    for tok in doc:
        if tok.text.strip()=="":
            continue
        lem.append(tok.lemma_)
    return lem


In [16]:
# @title Cargar BETO y definir funciones de embeddings
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", DEVICE)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModel.from_pretrained(MODEL_NAME, output_hidden_states=True).to(DEVICE).eval()

@torch.no_grad()
def embed_sentence_cls(textos: List[str] | str) -> np.ndarray:
    """
    Embedding CLS (promedio últimas 4 capas), normalizado L2.
    Acepta lista o string. Devuelve (N,768) o (768,) si N=1.
    """
    single = False
    if isinstance(textos, str):
        textos = [textos]
        single = True

    vs = []
    bs = 64
    for i in tqdm(range(0, len(textos), bs), desc="Embeddings CLS (batch)"):
        batch = textos[i:i+bs]
        enc = tokenizer(batch, return_tensors='pt', padding=True, truncation=True, max_length=128).to(DEVICE)
        out = model(**enc)
        hs = out.hidden_states
        cls = torch.stack([hs[-1][:,0], hs[-2][:,0], hs[-3][:,0], hs[-4][:,0]], dim=0).mean(dim=0)  # (B,768)
        v = torch.nn.functional.normalize(cls, p=2, dim=1)
        vs.append(v.detach().cpu().numpy())
    V = np.vstack(vs) if vs else np.zeros((0,768), dtype=np.float32)
    return V[0] if (single and V.shape[0] == 1) else V

@torch.no_grad()
def embed_tokens_en_context(oracion: str, palabras: List[str]) -> List[np.ndarray]:
    """
    Para cada palabra visible en la oración, obtiene el embedding contextual
    promediando sus subpalabras (últimas 4 capas). Si no hay alineación,
    usa CLS de la oración como fallback.
    """
    enc = tokenizer(
        oracion,
        return_tensors="pt",
        return_offsets_mapping=True,
        add_special_tokens=True,
        truncation=True,
        max_length=256,
    )
    offs = enc.pop("offset_mapping")[0].tolist()
    enc_model = {k: v.to(DEVICE) for k, v in enc.items()}
    out = model(**enc_model)
    hs = out.hidden_states

    # spans de palabras por búsqueda secuencial
    spans_pal, idx = [], 0
    for w in palabras:
        j = oracion.lower().find(w.lower(), idx)
        if j == -1: j = idx
        spans_pal.append((j, j+len(w)))
        idx = j + len(w)

    token_vecs = []
    for (a, b) in spans_pal:
        sub_idxs = [i for i, (x, y) in enumerate(offs) if not (x == 0 and y == 0) and (max(a, x) < min(b, y))]
        if not sub_idxs:
            v = embed_sentence_cls(oracion)  # (768,)
            token_vecs.append(v)
            continue
        sub_stack = []
        for layer in (-1, -2, -3, -4):
            layer_out = hs[layer].squeeze(0)  # (seq,768)
            sel = layer_out[sub_idxs, :].mean(dim=0)
            sub_stack.append(sel)
        v = torch.stack(sub_stack, dim=0).mean(dim=0)
        v = torch.nn.functional.normalize(v, p=2, dim=0)
        token_vecs.append(v.detach().cpu().numpy())
    return token_vecs


Dispositivo: cuda


Some weights of BertModel were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [17]:
# @title Indexar pictogramas y construir embeddings de etiquetas + mapa "unigrama exacto"
def cargar_pictos_y_indexar(dir_pictos: str):
    """
    Retorna:
      - etiquetas_txt[id] -> etiqueta original (con _)
      - etiquetas_norm[id] -> etiqueta normalizada (espacios)
      - idx_token[token] -> set(ids) por palabra completa
      - idx_frase[frase_norm] -> set(ids) por frase completa (multi-palabra)
      - exacto_unigrama_map[token] -> set(ids) cuyo nombre normalizado es exactamente ese token y tiene UNA palabra
    """
    etiquetas_txt: Dict[str, str] = {}
    etiquetas_norm: Dict[str, str] = {}
    idx_token: Dict[str, Set[str]] = defaultdict(set)
    idx_frase: Dict[str, Set[str]] = defaultdict(set)
    exacto_unigrama_map: Dict[str, Set[str]] = defaultdict(set)

    files = [f for f in os.listdir(dir_pictos) if f.lower().endswith((".jpg",".jpeg",".png"))]
    from tqdm.auto import tqdm
    for fname in tqdm(files, desc="Indexando pictogramas"):
        try:
            pid, name = fname.split("-", 1)
            int(pid)
        except:
            continue
        etiqueta = name.rsplit(".",1)[0]  # sin extensión
        et_norm = norm_text(etiqueta)

        etiquetas_txt[pid] = etiqueta
        etiquetas_norm[pid] = et_norm

        toks = tokens_de_etiqueta(et_norm)
        for t in toks:
            idx_token[t].add(pid)

        if " " in et_norm:
            idx_frase[et_norm].add(pid)

        # mapa exacto unigrama
        if len(toks) == 1 and toks[0] == et_norm:
            exacto_unigrama_map[et_norm].add(pid)

    return etiquetas_txt, etiquetas_norm, idx_token, idx_frase, exacto_unigrama_map

def construir_embeddings_etiquetas(etiquetas_txt: Dict[str,str], cache_path: str|None=None) -> Dict[str, np.ndarray]:
    """ Construye embeddings CLS por etiqueta en batch y cachea en .npy. """
    emb: Dict[str, np.ndarray] = {}
    if cache_path and os.path.exists(cache_path):
        try:
            loaded = np.load(cache_path, allow_pickle=True).item()
            emb = {k: np.array(v) for k, v in loaded.items()}
            print(f"Cargado cache embeddings: {cache_path} (#{len(emb)})")
        except Exception as e:
            print("Cache inválido, se reconstruirá:", e)

    faltan = [pid for pid in etiquetas_txt if pid not in emb]
    if faltan:
        textos = [etiquetas_txt[pid].replace("_"," ") for pid in faltan]
        V = embed_sentence_cls(textos)  # (N,768)
        for pid, v in zip(faltan, V):
            emb[pid] = v
        if cache_path:
            np.save(cache_path, emb, allow_pickle=True)
            print(f"Embeddings guardados en cache: {cache_path} (#{len(emb)})")
    return emb

# Ejecutar indexado + embeddings
etiquetas_txt, etiquetas_norm, idx_token, idx_frase, exacto_unigrama_map = cargar_pictos_y_indexar(RUTA_PICTOS)
emb_etq = construir_embeddings_etiquetas(etiquetas_txt, CACHE_EMB_PICTOS)
print(f"Total pictos indexados: {len(etiquetas_txt)}")


Indexando pictogramas:   0%|          | 0/13527 [00:00<?, ?it/s]

Embeddings CLS (batch):   0%|          | 0/212 [00:00<?, ?it/s]

Embeddings guardados en cache: /content/emb_pictos_v3.npy (#13527)
Total pictos indexados: 13527


In [18]:
# @title Detección de n-gramas, pool léxico y filtros de sensibilidad
def detectar_ngramas(oracion_norm_toks: List[str],
                     idx_frase: Dict[str,Set[str]]) -> List[Tuple[int,int,str,List[str]]]:
    """
    Retorna spans [(i_ini, i_fin_excl, frase_norm, ids)] ordenados y NO solapados.
    Ventanas 4,3,2. Acepta exacto si está en idx_frase o si es semilla muy parecida.
    """
    n = len(oracion_norm_toks)
    spans = []
    ocupado = [False]*n
    frases_bd = set(idx_frase.keys())

    for L in [4,3,2]:
        i = 0
        while i <= n - L:
            if any(ocupado[i:i+L]):
                i += 1; continue
            frag_norm = " ".join(oracion_norm_toks[i:i+L])
            ids = None

            if frag_norm in frases_bd:
                ids = list(idx_frase[frag_norm])
            elif frag_norm in SEMILLAS_NGRAM:
                # buscar la frase más similar en BD
                mejor, best = None, 0
                for f in frases_bd:
                    r = fuzz.ratio(frag_norm, f)
                    if r > best:
                        best, mejor = r, f
                if best >= 90 and mejor is not None:
                    ids = list(idx_frase[mejor])
                    frag_norm = mejor

            if ids:
                spans.append((i, i+L, frag_norm, ids))
                for k in range(i, i+L):
                    ocupado[k] = True
                i += L
            else:
                i += 1

    spans.sort(key=lambda x: x[0])
    return spans

def _token_pasa_blocklist(etiqueta_norm: str, dominio: str|None) -> bool:
    """ True si la etiqueta NO contiene términos bloqueados. """
    bl = set(BLOCKLIST_GLOBAL)
    if dominio and dominio in BLOCKLIST_POR_DOMINIO:
        bl |= BLOCKLIST_POR_DOMINIO[dominio]
    en = etiqueta_norm
    return not any(bad in en for bad in bl)

def pool_lexico(token: str,
                idx_token: Dict[str,Set[str]],
                etiquetas_norm: Dict[str,str],
                M: int=M_POOL,
                dominio: str|None=None) -> List[str]:
    """
    Pool robusto por palabra completa:
    - Exacto por token
    - Sinónimos (exactos)
    - Fuzzy contra tokens (umbral alto) SOLO si len(token)>=LONG_MIN_FUZZY
    - Filtro de sensibilidad (blocklist)
    """
    t = norm_text(token)
    cand: Set[str] = set()

    # exacto por palabra completa
    if t in idx_token:
        cand |= idx_token[t]

    # sinónimos exactos
    for s in SINONIMOS_BASE.get(t, set()):
        if s in idx_token:
            cand |= idx_token[s]

    # fuzzy estricto (token-level) sólo si el token es suficientemente largo
    if len(t) >= LONG_MIN_FUZZY and len(cand) < M:
        scored = []
        for pid, en in etiquetas_norm.items():
            if not _token_pasa_blocklist(en, dominio):
                continue
            toks = tokens_de_etiqueta(en)
            best = max((fuzz.ratio(t, tok) for tok in toks), default=0)
            # umbral alto + control de longitud
            if best >= 92 and abs(len(t) - max((len(tok) for tok in toks), default=len(t))) <= 1:
                scored.append((best, pid))
        scored.sort(reverse=True)
        for _, pid in scored:
            cand.add(pid)
            if len(cand) >= M:
                break

    # aplicar blocklist también al conjunto exacto
    cand_filtrado = [pid for pid in cand if _token_pasa_blocklist(etiquetas_norm[pid], dominio)]
    return cand_filtrado[:max(M, len(cand_filtrado))]


In [19]:
# @title Scoring compuesto con coherencia contextual, dominio y unigrama exacto
def score_lexico_base(token_norm: str, etiqueta_norm: str) -> float:
    """
    Score léxico base por palabra completa + sinónimos + fuzzy.
    (El bono por exacto unigrama se añade aparte para priorizarlo)
    """
    t = token_norm
    toks = tokens_de_etiqueta(etiqueta_norm)
    if t in toks:
        return 1.0
    if t in SINONIMOS_BASE and any(s in toks for s in SINONIMOS_BASE[t]):
        return 0.9
    best = max((fuzz.ratio(t, tok) for tok in toks), default=0) / 100.0
    return min(best, 0.70)

def penalty_pos(token_norm: str, etiqueta_norm: str) -> float:
    e0 = tokens_de_etiqueta(etiqueta_norm)[:1]
    e0 = e0[0] if e0 else ""
    es_verbo_t = token_norm.endswith(("ar","er","ir")) or token_norm in {"ser","estar","haber","ir"}
    es_verbo_e = e0.endswith(("ar","er","ir"))
    return 1.0 if es_verbo_t and not es_verbo_e else 0.0

def construir_bigrama_set(tokens_norm: List[str]) -> Set[Tuple[str,str]]:
    # Considera allowlist (permitidas no se filtran)
    toks = [t for t in tokens_norm if t and ((t not in STOP_TOKENS) or (t in ALLOWLIST_STOPWORDS))]
    return set(zip(toks, toks[1:])) if len(toks) >= 2 else set()

def bonus_coocurrencia(etiqueta_norm: str, tokens_oracion_set: Set[str], token_actual_norm: str) -> float:
    etoks = set(tokens_de_etiqueta(etiqueta_norm))
    otros = tokens_oracion_set - {token_actual_norm}
    overlap = len(etoks & otros)
    return min(2, overlap) * W_COOCC  # máx 2*W_COOCC

def bonus_bigrama(etiqueta_norm: str, bigramas_oracion: Set[Tuple[str,str]]) -> float:
    etoks = tokens_de_etiqueta(etiqueta_norm)
    et_bi = set(zip(etoks, etoks[1:])) if len(etoks) >= 2 else set()
    return W_BIGRAM if len(et_bi & bigramas_oracion) > 0 else 0.0

def bonus_dominio(etiqueta_norm: str, dominio: str|None) -> float:
    if not dominio or dominio not in BOOST_DOMINIO:
        return 0.0
    etoks = set(tokens_de_etiqueta(etiqueta_norm))
    bd = set(BOOST_DOMINIO[dominio])
    overlap = len(etoks & bd)
    return min(2, overlap) * W_BOOST_DOM

def bonus_unigrama_exact(token_norm: str, etiqueta_norm: str, toks: List[str],
                         tokens_oracion_set: Set[str]) -> float:
    """
    Bono fuerte si la etiqueta es EXACTAMENTE el token y es unigrama.
    Bono adicional si hay relación fuerte con verbos típicos (beber/tomar/querer/comer + agua/sandia/etc.).
    """
    bonus = 0.0
    if len(toks) == 1 and toks[0] == etiqueta_norm and etiqueta_norm == token_norm:
        bonus += EXACT_UNIGRAM_BONUS     # empuja el exacto al tope
        # relaciones verbo-sustantivo frecuentes
        if token_norm in {"agua","leche","jugo","pan","comida","sandia"} and ({"beber","tomar","querer","comer"} & tokens_oracion_set):
            bonus += ADJ_VERB_NOUN_PAIR_BONUS
        if token_norm in {"baño"} and ({"ir","querer"} & tokens_oracion_set):
            bonus += ADJ_VERB_NOUN_PAIR_BONUS
    # preferencia por unigrama frente a compuestos largos
    if len(toks) == 1:
        bonus += UNIGRAM_PREFER_BONUS
    elif len(toks) >= 3:
        bonus -= LONG_COMPOUND_PENALTY
    # cabeza o cola
    if toks and (toks[0] == token_norm or toks[-1] == token_norm):
        bonus += HEAD_POS_BONUS
    return bonus

def combinar_score(sim_emb, sim_ctx, lex_base, pen, b_cooc, b_bi, b_dom, b_uni):
    # pesos base + bonos
    return (0.50*sim_emb + 0.25*sim_ctx + 0.15*lex_base - 0.04*pen) + b_cooc + b_bi + b_dom + b_uni


In [20]:
# @title Ranking Top-K y pipeline por oración (con inyección de exacto unigrama)
def topk_candidatos_para_token(token_vis: str,
                               token_norm: str,        # lemma norm
                               vec_token: np.ndarray,
                               vec_oracion_cls: np.ndarray,
                               pool_ids: List[str],
                               emb_etq: Dict[str,np.ndarray],
                               etiquetas_norm: Dict[str,str],
                               etiquetas_txt: Dict[str,str],
                               tokens_oracion_set: Set[str],
                               bigramas_oracion: Set[tuple],
                               dominio: str|None,
                               exacto_unigrama_map: Dict[str,Set[str]],
                               token_norm_visible: str = None,   # NUEVO
                               K: int=K_POR_TOKEN) -> List[str]:
    # --- Inyectar exactos unigrama por lemma y por superficie ---
    exactos = set(exacto_unigrama_map.get(token_norm, set()))
    if token_norm_visible:
        exactos |= set(exacto_unigrama_map.get(token_norm_visible, set()))

    pool_union = set(pool_ids) | exactos
    if not pool_union:
        return [""] if not OMITIR_STOPWORDS_EN_SALIDA else []

    scores = []
    for pid in pool_union:
        en = etiquetas_norm[pid]
        if not _token_pasa_blocklist(en, dominio):
            continue
        ve = emb_etq[pid]
        sim_emb = float(np.dot(vec_token, ve))         # token contextual vs etiqueta (CLS)
        sim_ctx = float(np.dot(vec_oracion_cls, ve))   # CLS oración vs etiqueta (CLS)
        lex_base = score_lexico_base(token_norm, en)
        pen = penalty_pos(token_norm, en)
        b_cooc = bonus_coocurrencia(en, tokens_oracion_set, token_norm)
        b_bi   = bonus_bigrama(en, bigramas_oracion)
        b_dom  = bonus_dominio(en, dominio)
        toks_en = tokens_de_etiqueta(en)
        b_uni  = bonus_unigrama_exact(token_norm, en, toks_en, tokens_oracion_set)
        s = combinar_score(sim_emb, sim_ctx, lex_base, pen, b_cooc, b_bi, b_dom, b_uni)
        scores.append((s, pid))

    if not scores:
        return [""] if not OMITIR_STOPWORDS_EN_SALIDA else []

    scores.sort(reverse=True, key=lambda x: x[0])
    if scores[0][0] < UMBRAL_SCORE:
        return [""] if not OMITIR_STOPWORDS_EN_SALIDA else []

    out = []
    for _, pid in scores[:K]:
        out.append(f"{pid}-{etiquetas_txt[pid].replace(' ', '_')}.jpg")
    return out or ([""] if not OMITIR_STOPWORDS_EN_SALIDA else [])

def candidatos_para_oracion(oracion_str: str,
                            etiquetas_txt, etiquetas_norm, idx_token, idx_frase, emb_etq,
                            exacto_unigrama_map,
                            dominio: str|None,
                            k_por_token=K_POR_TOKEN, m_pool=M_POOL):
    """
    Devuelve (tokens_visibles, dict token->lista de candidatos) con n-gramas, filtros y ranking integrados.
    """
    import re
    # Tokens visibles (según el XML)
    toks_vis = [t for t in re.split(r"\s+", oracion_str.strip()) if t]

    # Lemmas opcionales; si no, usa visibles normalizados
    toks_base = lematiza_tokens(toks_vis) if nlp is not None else toks_vis
    toks_norm_lemma = [norm_text(t) for t in toks_base]
    toks_norm_vis   = [norm_text(t) for t in toks_vis]   # forma visible normalizada
    vec_cls = embed_sentence_cls(oracion_str)
    vecs_tok = embed_tokens_en_context(oracion_str, toks_vis)

    # Sets de coherencia contextual (allowlist cuenta como contexto)
    tokens_oracion_set = set([t for t in toks_norm_lemma if t and ((t not in STOP_TOKENS) or (t in ALLOWLIST_STOPWORDS))])
    bigramas_oracion = construir_bigrama_set(toks_norm_lemma)

    n = len(toks_vis)
    candidatos_por_token: Dict[str, List[str]] = {}
    spans = detectar_ngramas(toks_norm_lemma, idx_frase)
    ocupado = [False]*n
    preasignados: Dict[int, List[str]] = {}

    # 1) N-gramas primero
    for (i0, i1, frase_norm, ids) in spans:
        vtok = vecs_tok[i0]
        pool = [pid for pid in ids if _token_pasa_blocklist(etiquetas_norm[pid], dominio)]
        topk = topk_candidatos_para_token(" ".join(toks_vis[i0:i1]), toks_norm_lemma[i0], vtok, vec_cls,
                                          pool, emb_etq, etiquetas_norm, etiquetas_txt,
                                          tokens_oracion_set, bigramas_oracion, dominio,
                                          exacto_unigrama_map,
                                          token_norm_visible=toks_norm_vis[i0],
                                          K=k_por_token)
        preasignados[i0] = topk
        for k in range(i0, i1):
            ocupado[k] = True
            if k != i0:
                candidatos_por_token[toks_vis[k]] = [] if OMITIR_STOPWORDS_EN_SALIDA else [""]

    # 2) Tokens simples
    for i, (tok_vis, tok_norm_lem, tok_norm_vis_) in enumerate(zip(toks_vis, toks_norm_lemma, toks_norm_vis)):
        if i in preasignados:
            candidatos_por_token[tok_vis] = preasignados[i]
            continue

        # stopwords / ocupados
        if ocupado[i]:
            if not OMITIR_STOPWORDS_EN_SALIDA and (norm_text(tok_vis) in ALLOWLIST_STOPWORDS):
                candidatos_por_token[tok_vis] = [""]
            continue

        if es_puntuacion(tok_vis) or es_stop(tok_vis):
            if (norm_text(tok_vis) in ALLOWLIST_STOPWORDS):
                # Está permitida: continúa como token normal (NO se salta)
                pass
            else:
                if not OMITIR_STOPWORDS_EN_SALIDA:
                    candidatos_por_token[tok_vis] = [""]
                continue

        # Pool robusto: usa AMBAS formas (lemma y visible)
        if len(tok_norm_lem) <= TOKEN_CORTO_EXACTO:
            pool = []
            for key in {tok_norm_lem, tok_norm_vis_}:
                if key in idx_token:
                    pool += list(idx_token[key])
                for s in SINONIMOS_BASE.get(key, set()):
                    if s in idx_token:
                        pool += list(idx_token[s])
            # inyecta exactos unigrama por ambas formas
            pool = list({
                *pool,
                *list(exacto_unigrama_map.get(tok_norm_lem, [])),
                *list(exacto_unigrama_map.get(tok_norm_vis_, [])),
            })
            pool = [pid for pid in pool if _token_pasa_blocklist(etiquetas_norm[pid], dominio)][:m_pool]
        else:
            pool = pool_lexico(tok_norm_lem, idx_token, etiquetas_norm, M=m_pool, dominio=dominio)
            # asegura exacto unigrama en ambas formas
            pool = list(
                set(pool)
                | set(exacto_unigrama_map.get(tok_norm_lem, set()))
                | set(exacto_unigrama_map.get(tok_norm_vis_, set()))
            )

        if not pool:
            if (norm_text(tok_vis) in ALLOWLIST_STOPWORDS) and FORZAR_INCLUSION_ALLOWLIST_VACIOS:
                candidatos_por_token[tok_vis] = [""]
            else:
                if not OMITIR_STOPWORDS_EN_SALIDA:
                    candidatos_por_token[tok_vis] = [""]
            continue

        topk = topk_candidatos_para_token(tok_vis, tok_norm_lem, vecs_tok[i], vec_cls,
                                          pool, emb_etq, etiquetas_norm, etiquetas_txt,
                                          tokens_oracion_set, bigramas_oracion, dominio,
                                          exacto_unigrama_map,
                                          token_norm_visible=tok_norm_vis_,
                                          K=k_por_token)
        if topk == [""] and OMITIR_STOPWORDS_EN_SALIDA:
            continue
        candidatos_por_token[tok_vis] = topk

    return toks_vis, candidatos_por_token


In [21]:
# @title Construir candidatos para un XML y guardar a JSON (con progreso)
def _inferir_dominio_desde_path(path_xml: str) -> str|None:
    p = path_xml.lower()
    if "salud" in p: return "salud"
    if "educacion" in p or "educación" in p: return "educacion"
    if "cultura" in p: return "cultura"
    return None

from tqdm.auto import tqdm

def construir_candidatos_json(path_xml: str, dir_pictos: str, path_out_json: str,
                              k_por_token=K_POR_TOKEN, m_pool=M_POOL, dominio: str|None=None):
    oraciones = leer_oraciones_xml(path_xml)
    dom = dominio or _inferir_dominio_desde_path(path_xml)

    salida = {"oraciones": [], "metadata": {
        "modelo": MODEL_NAME,
        "k_por_token": k_por_token,
        "m_pool": m_pool,
        "umbral_score": UMBRAL_SCORE,
        "omitir_stopwords_en_salida": OMITIR_STOPWORDS_EN_SALIDA,
        "allowlist_stopwords": sorted(list(ALLOWLIST_STOPWORDS)),
        "forzar_allowlist_vacios": FORZAR_INCLUSION_ALLOWLIST_VACIOS,
        "dominio": dom,
        "version": "v3.2 (unigrama exacto + allowlist)"
    }}

    for S in tqdm(oraciones, desc=f"Procesando {os.path.basename(path_xml)}"):
        toks, cand = candidatos_para_oracion(
            S, etiquetas_txt, etiquetas_norm, idx_token, idx_frase, emb_etq,
            exacto_unigrama_map,
            dominio=dom, k_por_token=k_por_token, m_pool=m_pool
        )
        salida["oraciones"].append({
            "oracion": S,
            "tokens": toks,
            "candidatos": cand
        })

    with open(path_out_json, "w", encoding="utf-8") as f:
        json.dump(salida, f, ensure_ascii=False, indent=2)
    print(f"Guardado: {path_out_json}  ({len(salida['oraciones'])} oraciones)")


In [22]:
# @title Ejecutar pipeline para las 3 categorías
construir_candidatos_json(RUTA_XML_SALUD,     RUTA_PICTOS, OUT_SALUD)
construir_candidatos_json(RUTA_XML_EDUCACION, RUTA_PICTOS, OUT_EDUCACION)
construir_candidatos_json(RUTA_XML_CULTURA,   RUTA_PICTOS, OUT_CULTURA)


Procesando oraciones_salud_final.xml:   0%|          | 0/503 [00:00<?, ?it/s]

Embeddings CLS (batch):   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings CLS (batch):   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings CLS (batch):   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings CLS (batch):   0%|          | 0/1 [00:00<?, ?it/s]

...

Embeddings CLS (batch):   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings CLS (batch):   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings CLS (batch):   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings CLS (batch):   0%|          | 0/1 [00:00<?, ?it/s]

Guardado: /content/candidatos_cultura_final.json  (498 oraciones)


In [23]:
# @title Inspección rápida de resultados
import random, json, re

def muestra(path_json: str, n=3):
    print("\nArchivo:", path_json)
    with open(path_json, "r", encoding="utf-8") as f:
        data = json.load(f)
    total = len(data["oraciones"])
    idxs = random.sample(range(total), k=min(n,total))
    for i in idxs:
        o = data["oraciones"][i]
        print("\n--- Oración:", o["oracion"])
        for t in o["tokens"]:
            if "candidatos" in o and t in o["candidatos"]:
                cands = o["candidatos"][t]
                print(f"  - {t:>12s} -> {cands[:5]}")
            else:
                print(f"  - {t:>12s} -> (omitido por stopword/umbral)")

# Test específico en memoria (sin tocar JSONs): "yo comer sandia"
def prueba_sandia(oracion="yo comer sandia", dominio="cultura"):
    toks_vis = [t for t in re.split(r"\s+", oracion.strip()) if t]
    toks_base = lematiza_tokens(toks_vis) if nlp is not None else toks_vis
    toks_norm_lemma = [norm_text(t) for t in toks_base]
    toks_norm_vis   = [norm_text(t) for t in toks_vis]
    vec_cls = embed_sentence_cls(oracion)
    vecs_tok = embed_tokens_en_context(oracion, toks_vis)
    tokens_oracion_set = set([t for t in toks_norm_lemma if t and ((t not in STOP_TOKENS) or (t in ALLOWLIST_STOPWORDS))])
    bigramas_oracion = set(zip(toks_norm_lemma, toks_norm_lemma[1:])) if len(toks_norm_lemma)>=2 else set()

    print("\nPrueba: ", oracion)
    for i, (tok_vis, tok_norm_lem, tok_norm_vis_) in enumerate(zip(toks_vis, toks_norm_lemma, toks_norm_vis)):
        if es_puntuacion(tok_vis):
            print(f"  - {tok_vis:>12s} -> (puntuación)")
            continue
        # construir pool combinado (lemma + visible)
        if len(tok_norm_lem) <= 3:
            pool = []
            for key in {tok_norm_lem, tok_norm_vis_}:
                if key in idx_token:
                    pool += list(idx_token[key])
                for s in SINONIMOS_BASE.get(key, set()):
                    if s in idx_token:
                        pool += list(idx_token[s])
            pool = list({
                *pool,
                *list(exacto_unigrama_map.get(tok_norm_lem, [])),
                *list(exacto_unigrama_map.get(tok_norm_vis_, [])),
            })
            pool = [pid for pid in pool if _token_pasa_blocklist(etiquetas_norm[pid], dominio)][:M_POOL]
        else:
            pool = pool_lexico(tok_norm_lem, idx_token, etiquetas_norm, M=M_POOL, dominio=dominio)
            pool = list(
                set(pool)
                | set(exacto_unigrama_map.get(tok_norm_lem, set()))
                | set(exacto_unigrama_map.get(tok_norm_vis_, set()))
            )
        topk = topk_candidatos_para_token(
            tok_vis, tok_norm_lem, vecs_tok[i], vec_cls, pool, emb_etq, etiquetas_norm, etiquetas_txt,
            tokens_oracion_set, bigramas_oracion, dominio, exacto_unigrama_map,
            token_norm_visible=tok_norm_vis_, K=K_POR_TOKEN
        )
        print(f"  - {tok_vis:>12s} -> {topk[:5]}")

muestra(OUT_SALUD, n=3);
muestra(OUT_EDUCACION, n=3);
muestra(OUT_CULTURA, n=3)
#prueba_sandia("yo comer sandia", dominio="cultura")



Archivo: /content/candidatos_salud_final.json

--- Oración: Tengo los labios secos
  -        Tengo -> ['8978-tener.jpg', '2617-tener.jpg', '4988-tener_tiempo.jpg', '2610-tener_hambre.jpg', '10007-tener_frío.jpg']
  -          los -> ['2046-él.jpg', '8434-él.jpg', '2062-él.jpg', '3058-los.jpg', '5253-el_gato_con_botas.jpg']
  -       labios -> ['666-labios.jpg']
  -        secos -> ['1300-seco.jpg', '13512-tomate_seco.jpg']

--- Oración: Ver fotos de familia
  -          Ver -> ['2132-ver.jpg', '226-ver.jpg', '7116-ver.jpg', '7331-ver_los_fuegos_artificiales.jpg', '501-ver_la_televisión.jpg']
  -        fotos -> ['6962-foto.jpg', '2454-foto.jpg', '10820-foto_de_perfil.jpg', '6990-foto_de_perfil.jpg']
  -           de -> ['2424-de.jpg', '11401-disco_de_ostomía.jpg', '3049-guía_de_plantas.jpg', '4912-lata_de_aceite.jpg', '8847-ir_en_silla_de_ruedas.jpg']
  -      familia -> ['5680-familia.jpg', '8418-familia.jpg', '4261-familia.jpg', '4259-familia.jpg', '11360-familia.jpg']

--- Oración